<a href="https://colab.research.google.com/github/NileshPatil24-a/AntroAI/blob/main/fixed_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import numpy as np
import matplotlib.pyplot as plt
import json

In [ ]:
def json_to_data(link):
  json_data = []
  with open(link, 'r') as f:
     for line in f:
         json_data.append(json.loads(line))

  instruction_data = []
  for i in json_data:
    instruction_data.append(i['instruction'])

  response_data = []
  for i in json_data:
    response_data.append(i['response'])

  data = pd.DataFrame()
  data["instruction"] = instruction_data
  data['response'] = response_data

  return data

In [ ]:
file_1_data = json_to_data("/content/astrophysics_dataset.jsonl")

In [ ]:
file_2_data = json_to_data("/content/astrophysics_kachelrie_dataset.jsonl")

In [ ]:
combined_data = pd.concat([file_1_data, file_2_data], ignore_index=True)

In [ ]:
combined_data.shape

(65, 2)

In [ ]:
combined_data.head()

,instruction,response
0,State and explain Kepler's Laws. Show any key ...,Kepler's three laws describe how planets orbit...
1,Explain General solution to the two-body probl...,The two-body problem asks: given two masses in...
2,Explain Conic sections in astrophysics. Includ...,Conic sections are the four curve types formed...
3,Explain The elliptical orbit in astrophysics. ...,"An elliptical orbit is a closed, bound path wh..."
4,Explain Center of mass system in astrophysics....,The center of mass (CoM) frame simplifies the ...


In [ ]:
combined_data['response'][0].rstrip()

"Kepler's three laws describe how planets orbit the Sun, derived empirically from Tycho Brahe's observations.\n\nStep 1 — Law of Ellipses: Every planet orbits the Sun in an ellipse, with the Sun at one focus. The orbit shape is described by r = a(1-e²)/(1 + e·cos f), where a = semimajor axis, e = eccentricity, f = true anomaly.\n\nStep 2 — Law of Equal Areas: The radius vector sweeps equal areas in equal times. Mathematically, dA/dt = h/2 = constant, where h = r²·(df/dt) is the specific angular momentum. This means planets move faster when closer to the Sun.\n\nStep 3 — Law of Periods: The square of the orbital period equals the cube of the semimajor axis:\n  P² = a³\nwhere P is in years and a is in AU. Newton later refined this to:\n  P² = (4π²/G(M+m)) · a³\n\nFinal Answer: These laws unify under Newton's gravity — the ellipse shape comes from an inverse-square force law, equal areas from angular momentum conservation, and the period-axis relation from energy considerations."

In [ ]:
combined_data['response'] = combined_data['response'].apply(lambda x: x.rstrip())
combined_data['instruction'] =combined_data['instruction'].apply(lambda x: x.rstrip())

In [ ]:
combined_data['response'][0]

"Kepler's three laws describe how planets orbit the Sun, derived empirically from Tycho Brahe's observations.\n\nStep 1 — Law of Ellipses: Every planet orbits the Sun in an ellipse, with the Sun at one focus. The orbit shape is described by r = a(1-e²)/(1 + e·cos f), where a = semimajor axis, e = eccentricity, f = true anomaly.\n\nStep 2 — Law of Equal Areas: The radius vector sweeps equal areas in equal times. Mathematically, dA/dt = h/2 = constant, where h = r²·(df/dt) is the specific angular momentum. This means planets move faster when closer to the Sun.\n\nStep 3 — Law of Periods: The square of the orbital period equals the cube of the semimajor axis:\n  P² = a³\nwhere P is in years and a is in AU. Newton later refined this to:\n  P² = (4π²/G(M+m)) · a³\n\nFinal Answer: These laws unify under Newton's gravity — the ellipse shape comes from an inverse-square force law, equal areas from angular momentum conservation, and the period-axis relation from energy considerations."

In [ ]:
def create_prompt(row):
    return f"""### Instruction:
{row['instruction']}

### Response:
{row['response']}"""

In [ ]:
combined_data["text"] = combined_data.apply(create_prompt, axis=1)

In [ ]:
combined_data.head()

,instruction,response,text
0,State and explain Kepler's Laws. Show any key ...,Kepler's three laws describe how planets orbit...,### Instruction:\nState and explain Kepler's L...
1,Explain General solution to the two-body probl...,The two-body problem asks: given two masses in...,### Instruction:\nExplain General solution to ...
2,Explain Conic sections in astrophysics. Includ...,Conic sections are the four curve types formed...,### Instruction:\nExplain Conic sections in as...
3,Explain The elliptical orbit in astrophysics. ...,"An elliptical orbit is a closed, bound path wh...",### Instruction:\nExplain The elliptical orbit...
4,Explain Center of mass system in astrophysics....,The center of mass (CoM) frame simplifies the ...,### Instruction:\nExplain Center of mass syste...


In [ ]:
def create_prompt(row):
    return f"""### Instruction:
{row['instruction']}

### Response:
{row['response']}"""

combined_data["text"] = combined_data.apply(create_prompt, axis=1)

In [ ]:
print(combined_data["text"].iloc[0])

### Instruction:
State and explain Kepler's Laws. Show any key equations.

### Response:
Kepler's three laws describe how planets orbit the Sun, derived empirically from Tycho Brahe's observations.

Step 1 — Law of Ellipses: Every planet orbits the Sun in an ellipse, with the Sun at one focus. The orbit shape is described by r = a(1-e²)/(1 + e·cos f), where a = semimajor axis, e = eccentricity, f = true anomaly.

Step 2 — Law of Equal Areas: The radius vector sweeps equal areas in equal times. Mathematically, dA/dt = h/2 = constant, where h = r²·(df/dt) is the specific angular momentum. This means planets move faster when closer to the Sun.

Step 3 — Law of Periods: The square of the orbital period equals the cube of the semimajor axis:
  P² = a³
where P is in years and a is in AU. Newton later refined this to:
  P² = (4π²/G(M+m)) · a³

Final Answer: These laws unify under Newton's gravity — the ellipse shape comes from an inverse-square force law, equal areas from angular momentum con

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(combined_data[["text"]])

In [ ]:
import subprocess
import sys

# Install bitsandbytes before loading the model
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'])
print('bitsandbytes installed successfully')

bitsandbytes installed successfully


In [ ]:
import importlib
import bitsandbytes  # ensure fresh import after install
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
def tokenize_function(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/65 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import TrainingArguments, Trainer

# Only use fp16 if CUDA is available; use bf16 on TPU or disable on CPU
use_fp16 = torch.cuda.is_available()
use_bf16 = (not use_fp16) and (torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

training_args = TrainingArguments(
    output_dir="./astro_model",
    per_device_train_batch_size=1,
    num_train_epochs=10,
    logging_steps=1,
    save_strategy="no",
    learning_rate=2e-4,
    fp16=use_fp16,
    bf16=use_bf16,
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Step,Training Loss
1,2.237027
2,1.928406
3,1.938303
4,1.900093
5,2.074056
6,2.064825
7,2.200897
8,2.260525
9,2.041360
10,1.955990


TrainOutput(global_step=650, training_loss=1.3684434146147508, metrics={'train_runtime': 268.9959, 'train_samples_per_second': 2.416, 'train_steps_per_second': 2.416, 'total_flos': 2067963523891200.0, 'train_loss': 1.3684434146147508, 'epoch': 10.0})

In [ ]:
def generate_response(model, tokenizer, instruction, max_new_tokens=300):
    """
    Run inference on a single instruction prompt.
    Returns the generated response string.
    """
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)



test_instruction = "Is Gravity ocuur in Space"
response = generate_response(model, tokenizer, test_instruction)
print(response)


Gravity is a fundamental force of nature, but it is not directly observable. It is only felt by objects moving in space-time.

In the Earth-Moon system, the Moon's gravitational pull on the Earth is 3.89×10⁻⁶ m/s² (1.4 m/s), which is small compared to the Earth's gravitational pull on the Moon (1.2 m/s²).

The gravitational constant is determined by the mass of the Earth (9.806 m/s²), so the gravitational constant of the Moon (9.806 m/s²/1.4) is much smaller than the Earth's.

The Moon's gravitational field is felt by the Earth, but the Earth's gravitational field is felt by the Moon. The Earth's gravitational field is felt by all matter in space, while the Moon's gravitational field is felt only by matter moving in space-time.

Finally, the gravitational constant is related to the mass of the universe (9.12×10²¹ kg/m³) via the Planck length (a=ħc/2).


In [ ]:
# Zip and download your trained model
import shutil
shutil.make_archive('/content/astro_model', 'zip', '/content/astro_model')

from google.colab import files
files.download('/content/astro_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>